In [ ]:
# Import necessary libraries
import os
import sys
import json
import numpy as np
import cv2
import torch
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import openai
from typing import List, Dict, Optional, Tuple

# Add current directory to path for pipeline imports
sys.path.append('.')
sys.path.append('..')

# Import pipeline components
from pipeline import GSAMDetector, ImageVisualizer
from pipeline.data_loader import CustomDirectoryDataLoader

# Setup matplotlib for inline plotting
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 10

print("✅ Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA device: {torch.cuda.get_device_name()}")


In [ ]:
# Configuration - Update these paths according to your setup
CONFIG = {
    # Model paths
    'grounding_config_file': 'GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py',
    'grounding_checkpoint': 'weight/groundingdino_swint_ogc.pth',
    'sam_version': 'vit_h',
    'sam_checkpoint': 'weight/sam_vit_h_4b8939.pth',
    'sam_hq_checkpoint': None,
    'use_sam_hq': False,
    
    # Detection thresholds
    'box_threshold': 0.3,
    'text_threshold': 0.25,
    'nms_threshold': 0.5,
    
    # Device
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    
    # BERT path (optional)
    'bert_base_uncased_path': None,
    
    # Part filtering
    'min_area_ratio': 0.005,
    'max_area_ratio': 0.5,
}

# Test image path - you can change this to your own image
TEST_IMAGE_PATH = "../data/eval_coco_animals"  # Update this path

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")
    
# Check if OpenAI API key is set (needed for vocabulary generation)
if not os.getenv('OPENAI_API_KEY'):
    print("⚠️  Warning: OPENAI_API_KEY not set. You'll need to use predefined vocabularies.")


In [ ]:
# Initialize GSAM Detector
print("🚀 Initializing GSAM Detector...")

try:
    # Initialize OpenAI client if API key is available
    openai_client = openai.OpenAI() if os.getenv('OPENAI_API_KEY') else None
    
    # Initialize GSAM detector
    gsam_detector = GSAMDetector(
        grounding_config_file=CONFIG['grounding_config_file'],
        grounding_checkpoint=CONFIG['grounding_checkpoint'],
        sam_version=CONFIG['sam_version'],
        sam_checkpoint=CONFIG['sam_checkpoint'],
        sam_hq_checkpoint=CONFIG['sam_hq_checkpoint'],
        use_sam_hq=CONFIG['use_sam_hq'],
        box_threshold=CONFIG['box_threshold'],
        text_threshold=CONFIG['text_threshold'],
        bert_base_uncased_path=CONFIG['bert_base_uncased_path'],
        device=CONFIG['device'],
        openai_client=openai_client
    )
    
    # Initialize visualizer
    visualizer = ImageVisualizer()
    
    print("✅ GSAM Detector initialized successfully!")
    print(f"   Device: {CONFIG['device']}")
    print(f"   GroundingDINO: {os.path.basename(CONFIG['grounding_checkpoint'])}")
    print(f"   SAM: {CONFIG['sam_version']}")
    
except Exception as e:
    print(f"❌ Error initializing GSAM Detector: {e}")
    print("Please check your model paths in the CONFIG section.")


In [ ]:
# Helper functions for image loading and visualization

def load_image_from_path(image_path: str) -> np.ndarray:
    """Load image from path and convert to RGB numpy array"""
    if not os.path.exists(image_path):
        raise FileNotFoundError(f"Image not found: {image_path}")
    
    # Load image using PIL and convert to RGB
    pil_image = Image.open(image_path).convert('RGB')
    # Convert to numpy array
    img_array = np.array(pil_image)
    return img_array

def display_image(img_array: np.ndarray, title: str = "Image", figsize: Tuple = (10, 8)):
    """Display image using matplotlib"""
    plt.figure(figsize=figsize)
    plt.imshow(img_array)
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def list_available_images(directory: str, extensions: List[str] = ['.jpg', '.jpeg', '.png', '.bmp']) -> List[str]:
    """List available images in a directory"""
    if not os.path.exists(directory):
        return []
    
    images = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if any(file.lower().endswith(ext) for ext in extensions):
                images.append(os.path.join(root, file))
    return sorted(images)

# List available test images
if os.path.exists(TEST_IMAGE_PATH):
    available_images = list_available_images(TEST_IMAGE_PATH)
    print(f"📁 Found {len(available_images)} images in {TEST_IMAGE_PATH}")
    if available_images:
        print("First 5 images:")
        for i, img_path in enumerate(available_images[:5]):
            print(f"  {i+1}. {os.path.basename(img_path)}")
else:
    print(f"⚠️  Test image directory not found: {TEST_IMAGE_PATH}")
    print("Please update TEST_IMAGE_PATH in the configuration cell.")


In [ ]:
# Load a test image
# Option 1: Load from available images list
if 'available_images' in locals() and available_images:
    # Use the first available image, or modify the index
    image_index = 0  # Change this to try different images
    if image_index < len(available_images):
        selected_image_path = available_images[image_index]
    else:
        selected_image_path = available_images[0]
        print(f"Index {image_index} out of range, using first image instead")
else:
    # Option 2: Specify a direct path to your image
    selected_image_path = "path/to/your/image.jpg"  # Update this path

print(f"Selected image: {selected_image_path}")

# Load and display the image
try:
    test_image = load_image_from_path(selected_image_path)
    print(f"✅ Image loaded successfully!")
    print(f"   Shape: {test_image.shape}")
    print(f"   Size: {test_image.shape[1]}x{test_image.shape[0]} pixels")
    
    # Display the original image
    display_image(test_image, f"Original Image: {os.path.basename(selected_image_path)}")
    
except Exception as e:
    print(f"❌ Error loading image: {e}")
    print("Please update the image path in this cell.")


In [ ]:
# Vocabulary Generation and Handling

# Predefined vocabularies for different object types
PREDEFINED_VOCABULARIES = {
    'animal': [
        'animal head', 'animal body', 'animal leg', 'animal tail', 'animal ear', 
        'animal eye', 'animal nose', 'animal mouth', 'animal paw', 'animal fur'
    ],
    'person': [
        'person head', 'person body', 'person arm', 'person leg', 'person hand',
        'person face', 'person hair', 'person clothes', 'person shoe', 'person torso'
    ],
    'vehicle': [
        'vehicle wheel', 'vehicle door', 'vehicle window', 'vehicle headlight',
        'vehicle bumper', 'vehicle mirror', 'vehicle roof', 'vehicle tire'
    ],
    'furniture': [
        'furniture leg', 'furniture surface', 'furniture handle', 'furniture back',
        'furniture arm', 'furniture cushion', 'furniture drawer'
    ],
    'general': [
        'object part', 'surface', 'edge', 'corner', 'handle', 'button',
        'component', 'section', 'detail', 'feature'
    ]
}

# Choose vocabulary method
USE_PREDEFINED_VOCAB = True  # Set to False to use OpenAI generation
VOCAB_TYPE = 'animal'  # Choose from: animal, person, vehicle, furniture, general
ARTIFACT_TYPE = 'distortion'  # Choose from: addition, removal, distortion

if USE_PREDEFINED_VOCAB:
    if VOCAB_TYPE in PREDEFINED_VOCABULARIES:
        vocabulary = PREDEFINED_VOCABULARIES[VOCAB_TYPE]
        print(f"📝 Using predefined vocabulary for '{VOCAB_TYPE}':")
        for i, term in enumerate(vocabulary):
            print(f"   {i+1}. {term}")
    else:
        vocabulary = PREDEFINED_VOCABULARIES['general']
        print(f"⚠️  Vocab type '{VOCAB_TYPE}' not found, using 'general'")
else:
    # Generate vocabulary using OpenAI (if available)
    if openai_client and 'test_image' in locals():
        print(f"🤖 Generating vocabulary using OpenAI for artifact type: {ARTIFACT_TYPE}")
        try:
            vocabulary = gsam_detector.generate_subpart_vocab(test_image, ARTIFACT_TYPE)
            print(f"✅ Generated vocabulary ({len(vocabulary)} terms):")
            for i, term in enumerate(vocabulary):
                print(f"   {i+1}. {term}")
        except Exception as e:
            print(f"❌ Error generating vocabulary: {e}")
            print("Falling back to predefined vocabulary...")
            vocabulary = PREDEFINED_VOCABULARIES.get(VOCAB_TYPE, PREDEFINED_VOCABULARIES['general'])
    else:
        print("⚠️  OpenAI client not available or image not loaded. Using predefined vocabulary.")
        vocabulary = PREDEFINED_VOCABULARIES.get(VOCAB_TYPE, PREDEFINED_VOCABULARIES['general'])

print(f"\n🎯 Final vocabulary: {len(vocabulary)} terms")
print(f"   Artifact type: {ARTIFACT_TYPE}")
print(f"   Method: {'Predefined' if USE_PREDEFINED_VOCAB else 'OpenAI Generated'}")


In [ ]:
# Run GSAM Detection Pipeline

if 'test_image' in locals() and 'vocabulary' in locals():
    print("🔍 Running GSAM detection pipeline...")
    print(f"   Image shape: {test_image.shape}")
    print(f"   Vocabulary: {len(vocabulary)} terms")
    print(f"   Device: {CONFIG['device']}")
    
    try:
        # Step 1: Detect parts using Grounded SAM
        print("\n1️⃣ Detecting parts with GroundingDINO + SAM...")
        predictions, visualized_output = gsam_detector.detect_parts(test_image, vocabulary)
        
        print(f"✅ Detection completed!")
        print(f"   Found {len(predictions['pred_boxes'])} detections")
        
        # Display basic statistics
        if len(predictions['pred_boxes']) > 0:
            scores = predictions['scores'].cpu().numpy()
            print(f"   Score range: {scores.min():.3f} - {scores.max():.3f}")
            print(f"   Mean score: {scores.mean():.3f}")
            
            # Show top detections
            top_indices = np.argsort(scores)[::-1][:5]  # Top 5 by score
            print("\n   Top 5 detections:")
            for i, idx in enumerate(top_indices):
                class_idx = predictions['pred_classes'][idx].item()
                score = scores[idx]
                if class_idx < len(vocabulary):
                    class_name = vocabulary[class_idx]
                    print(f"     {i+1}. {class_name} (score: {score:.3f})")
        
        # Step 2: Sample target part (following the original pipeline)
        print("\n2️⃣ Sampling target part...")
        sampled_instance, sampled_idx, class_name = gsam_detector.sample_target_part(
            predictions, vocabulary, CONFIG['min_area_ratio'], CONFIG['max_area_ratio']
        )
        
        if sampled_instance is not None:
            print(f"✅ Target part sampled!")
            print(f"   Class: {class_name}")
            print(f"   Index: {sampled_idx}")
            print(f"   Score: {sampled_instance['score'].item():.3f}")
            bbox = sampled_instance['pred_box'].cpu().numpy()
            print(f"   Bbox: [{bbox[0]:.1f}, {bbox[1]:.1f}, {bbox[2]:.1f}, {bbox[3]:.1f}]")
        else:
            print("⚠️  No suitable target part found (may be filtered by area ratio)")
            sampled_instance, sampled_idx, class_name = None, None, None
            
    except Exception as e:
        print(f"❌ Error during detection: {e}")
        import traceback
        traceback.print_exc()
        predictions, visualized_output = None, None
        sampled_instance, sampled_idx, class_name = None, None, None

else:
    print("⚠️  Please ensure both test_image and vocabulary are loaded first!")


In [ ]:
# Visualization of Detection Results

def visualize_detections_matplotlib(image: np.ndarray, predictions: Dict, vocabulary: List[str], 
                                  sampled_idx: Optional[int] = None, max_detections: int = 20):
    """Visualize detection results using matplotlib"""
    
    if len(predictions['pred_boxes']) == 0:
        print("No detections to visualize")
        return
    
    # Limit number of detections for clarity
    num_dets = min(len(predictions['pred_boxes']), max_detections)
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))
    
    # Left: Original image with all detections
    ax1.imshow(image)
    ax1.set_title(f"All Detections (Top {num_dets})", fontsize=14, fontweight='bold')
    ax1.axis('off')
    
    # Sort by score and take top detections
    scores = predictions['scores'].cpu().numpy()
    top_indices = np.argsort(scores)[::-1][:num_dets]
    
    colors = plt.cm.Set3(np.linspace(0, 1, num_dets))
    
    for i, idx in enumerate(top_indices):
        bbox = predictions['pred_boxes'][idx].cpu().numpy()
        class_idx = predictions['pred_classes'][idx].item()
        score = scores[idx]
        
        # Draw bounding box
        rect = patches.Rectangle(
            (bbox[0], bbox[1]), bbox[2] - bbox[0], bbox[3] - bbox[1],
            linewidth=2, edgecolor=colors[i], facecolor='none'
        )
        ax1.add_patch(rect)
        
        # Add label
        if class_idx < len(vocabulary):
            label = f"{vocabulary[class_idx]}: {score:.2f}"
            ax1.text(bbox[0], bbox[1] - 5, label, fontsize=8, 
                    bbox=dict(boxstyle="round,pad=0.3", facecolor=colors[i], alpha=0.7))
    
    # Right: Highlighted sampled instance
    ax2.imshow(image)
    ax2.set_title("Sampled Target Part", fontsize=14, fontweight='bold')
    ax2.axis('off')
    
    if sampled_idx is not None and sampled_idx < len(predictions['pred_boxes']):
        bbox = predictions['pred_boxes'][sampled_idx].cpu().numpy()
        class_idx = predictions['pred_classes'][sampled_idx].item()
        score = scores[sampled_idx]
        
        # Draw highlighted bounding box
        rect = patches.Rectangle(
            (bbox[0], bbox[1]), bbox[2] - bbox[0], bbox[3] - bbox[1],
            linewidth=4, edgecolor='red', facecolor='none'
        )
        ax2.add_patch(rect)
        
        # Add label
        if class_idx < len(vocabulary):
            label = f"TARGET: {vocabulary[class_idx]} ({score:.3f})"
            ax2.text(bbox[0], bbox[1] - 10, label, fontsize=12, fontweight='bold',
                    bbox=dict(boxstyle="round,pad=0.5", facecolor='red', alpha=0.8, edgecolor='white'))
    else:
        ax2.text(0.5, 0.5, "No target part sampled", transform=ax2.transAxes, 
                ha='center', va='center', fontsize=16, 
                bbox=dict(boxstyle="round,pad=0.5", facecolor='yellow', alpha=0.8))
    
    plt.tight_layout()
    plt.show()

# Visualize results if detection was successful
if 'predictions' in locals() and predictions is not None:
    print("📊 Visualizing detection results...")
    
    # Method 1: Using matplotlib (detailed)
    visualize_detections_matplotlib(test_image, predictions, vocabulary, sampled_idx)
    
    # Method 2: Using the original visualized output (if available)
    if 'visualized_output' in locals() and visualized_output is not None:
        print("\n📊 GSAM Original Visualization:")
        display_image(visualized_output, "GSAM Detection Results")
        
else:
    print("⚠️  No detection results to visualize. Run the detection pipeline first!")


In [ ]:
# Mask Visualization and Analysis

def visualize_segmentation_masks(image: np.ndarray, predictions: Dict, sampled_idx: Optional[int] = None):
    """Visualize segmentation masks from GSAM"""
    
    if 'pred_masks' not in predictions or len(predictions['pred_masks']) == 0:
        print("⚠️  No segmentation masks found in predictions")
        return
    
    masks = predictions['pred_masks'].cpu().numpy()
    scores = predictions['scores'].cpu().numpy()
    
    # Show masks for top detections
    top_indices = np.argsort(scores)[::-1][:6]  # Top 6 for 2x3 grid
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    for i, idx in enumerate(top_indices):
        if i >= len(axes):
            break
            
        mask = masks[idx]
        score = scores[idx]
        
        # Create overlay
        overlay = image.copy()
        colored_mask = np.zeros_like(overlay)
        colored_mask[mask > 0] = [255, 0, 0]  # Red mask
        overlay = cv2.addWeighted(overlay, 0.7, colored_mask, 0.3, 0)
        
        axes[i].imshow(overlay)
        title = f"Mask {idx} (Score: {score:.3f})"
        if idx == sampled_idx:
            title += " - TARGET"
            axes[i].set_title(title, fontsize=12, fontweight='bold', color='red')
        else:
            axes[i].set_title(title, fontsize=10)
        axes[i].axis('off')
    
    # Hide unused subplots
    for i in range(len(top_indices), len(axes)):
        axes[i].axis('off')
    
    plt.tight_layout()
    plt.show()

def analyze_mask_properties(predictions: Dict, vocabulary: List[str], image_shape: Tuple):
    """Analyze properties of detected masks"""
    
    if 'pred_masks' not in predictions:
        print("No masks to analyze")
        return
    
    masks = predictions['pred_masks'].cpu().numpy()
    scores = predictions['scores'].cpu().numpy()
    classes = predictions['pred_classes'].cpu().numpy()
    
    print(f"🔍 Mask Analysis for {len(masks)} detections:")
    print(f"   Image size: {image_shape[1]}x{image_shape[0]} pixels")
    print(f"   Total area: {image_shape[0] * image_shape[1]:,} pixels")
    
    # Calculate areas and statistics
    mask_areas = []
    area_ratios = []
    
    total_image_area = image_shape[0] * image_shape[1]
    
    for i, mask in enumerate(masks):
        area = np.sum(mask > 0)
        ratio = area / total_image_area
        mask_areas.append(area)
        area_ratios.append(ratio)
    
    # Sort by score for display
    sorted_indices = np.argsort(scores)[::-1]
    
    print(f"\n📊 Top 10 Detections by Score:")
    print(f"{'Rank':<4} {'Class':<20} {'Score':<8} {'Area':<8} {'Ratio':<8}")
    print("-" * 60)
    
    for rank, idx in enumerate(sorted_indices[:10]):
        class_name = vocabulary[classes[idx]] if classes[idx] < len(vocabulary) else f"class_{classes[idx]}"
        score = scores[idx]
        area = mask_areas[idx]
        ratio = area_ratios[idx]
        
        print(f"{rank+1:<4} {class_name:<20} {score:<8.3f} {area:<8,} {ratio:<8.4f}")
    
    # Statistics
    print(f"\n📈 Overall Statistics:")
    print(f"   Mean area ratio: {np.mean(area_ratios):.4f}")
    print(f"   Std area ratio: {np.std(area_ratios):.4f}")
    print(f"   Min area ratio: {np.min(area_ratios):.4f}")
    print(f"   Max area ratio: {np.max(area_ratios):.4f}")
    print(f"   Filtering range: {CONFIG['min_area_ratio']:.4f} - {CONFIG['max_area_ratio']:.4f}")
    
    # Count masks within filtering range
    within_range = sum(1 for ratio in area_ratios 
                      if CONFIG['min_area_ratio'] <= ratio <= CONFIG['max_area_ratio'])
    print(f"   Masks within filter range: {within_range}/{len(masks)} ({within_range/len(masks)*100:.1f}%)")

# Run mask analysis if we have results
if 'predictions' in locals() and predictions is not None and 'test_image' in locals():
    
    # Analyze mask properties
    analyze_mask_properties(predictions, vocabulary, test_image.shape)
    
    # Visualize masks
    print(f"\n🎭 Mask Visualization:")
    visualize_segmentation_masks(test_image, predictions, sampled_idx)
    
else:
    print("⚠️  Run the detection pipeline first to generate masks!")


In [ ]:
# Interactive Experimentation

def run_gsam_experiment(image_path: str = None, custom_vocab: List[str] = None, 
                       box_threshold: float = None, text_threshold: float = None,
                       min_area: float = None, max_area: float = None):
    """Run a complete GSAM experiment with custom parameters"""
    
    print("🧪 Running GSAM Experiment")
    print("=" * 50)
    
    # Use provided parameters or defaults
    box_thresh = box_threshold or CONFIG['box_threshold']
    text_thresh = text_threshold or CONFIG['text_threshold']
    min_area_ratio = min_area or CONFIG['min_area_ratio']
    max_area_ratio = max_area or CONFIG['max_area_ratio']
    
    print(f"Parameters:")
    print(f"  Box threshold: {box_thresh}")
    print(f"  Text threshold: {text_thresh}")
    print(f"  Area ratio range: {min_area_ratio:.4f} - {max_area_ratio:.4f}")
    
    # Load image
    if image_path and os.path.exists(image_path):
        img = load_image_from_path(image_path)
        print(f"  Image: {os.path.basename(image_path)} ({img.shape})")
    elif 'test_image' in locals():
        img = test_image
        print(f"  Using loaded test image ({img.shape})")
    else:
        print("❌ No valid image provided or loaded")
        return
    
    # Use custom vocabulary or default
    vocab = custom_vocab or vocabulary
    print(f"  Vocabulary: {len(vocab)} terms")
    
    # Update detector thresholds temporarily
    original_box_thresh = gsam_detector.box_threshold
    original_text_thresh = gsam_detector.text_threshold
    gsam_detector.box_threshold = box_thresh
    gsam_detector.text_threshold = text_thresh
    
    try:
        # Run detection
        print("\n🔍 Running detection...")
        preds, viz_output = gsam_detector.detect_parts(img, vocab)
        
        # Sample target
        print("🎯 Sampling target...")
        sampled_inst, sampled_idx, class_name = gsam_detector.sample_target_part(
            preds, vocab, min_area_ratio, max_area_ratio
        )
        
        # Results summary
        print(f"\n📊 Results:")
        print(f"  Total detections: {len(preds['pred_boxes'])}")
        if sampled_inst is not None:
            print(f"  Sampled target: {class_name} (score: {sampled_inst['score'].item():.3f})")
        else:
            print(f"  No target sampled (filtered by area ratio)")
        
        # Quick visualization
        if len(preds['pred_boxes']) > 0:
            visualize_detections_matplotlib(img, preds, vocab, sampled_idx)
        
        return preds, sampled_inst, sampled_idx, class_name
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return None, None, None, None
    
    finally:
        # Restore original thresholds
        gsam_detector.box_threshold = original_box_thresh
        gsam_detector.text_threshold = original_text_thresh

# Example experiments you can run:
print("🧪 Example Experiments:")
print("=" * 50)
print("1. Try different thresholds:")
print("   run_gsam_experiment(box_threshold=0.25, text_threshold=0.2)")
print("")
print("2. Try custom vocabulary:")
print("   custom_vocab = ['head', 'body', 'leg', 'tail']")
print("   run_gsam_experiment(custom_vocab=custom_vocab)")
print("")
print("3. Try different area filtering:")
print("   run_gsam_experiment(min_area=0.01, max_area=0.3)")
print("")
print("4. Try different image:")
print("   if len(available_images) > 1:")
print("       run_gsam_experiment(image_path=available_images[1])")
print("")
print("Uncomment and run any of these experiments below:")


In [ ]:
# Quick Experiments - Uncomment any line to run

# 1. Try different detection thresholds (more sensitive)
# run_gsam_experiment(box_threshold=0.25, text_threshold=0.2)

# 2. Try custom vocabulary for specific objects
# custom_vocab = ['animal head', 'animal eye', 'animal nose', 'animal ear', 'animal mouth']
# run_gsam_experiment(custom_vocab=custom_vocab)

# 3. Try stricter area filtering (smaller parts only)
# run_gsam_experiment(min_area=0.005, max_area=0.1)

# 4. Try a different image (if available)
# if 'available_images' in locals() and len(available_images) > 1:
#     run_gsam_experiment(image_path=available_images[1])

# 5. Combination experiment
# run_gsam_experiment(
#     custom_vocab=['head', 'face', 'eye', 'nose', 'mouth'], 
#     box_threshold=0.2, 
#     min_area=0.001, 
#     max_area=0.2
# )

print("✅ Notebook ready! Uncomment any experiment above to try different configurations.")
print("💡 Tips:")
print("   - Lower thresholds = more detections (but possibly more noise)")
print("   - Custom vocabularies = more targeted detection")
print("   - Smaller area ratios = focus on fine details")
print("   - Larger area ratios = focus on major parts")
